# Phase 1: Text representation

## Research Question

How can text be converted into numerical representations that allow a computer to compare documents?

## Objectives

1. Load a text corpus
2. Tokenize text
3. Implement Bag of Words
4. Implement TF - IDF (Term Frequency - Inverse Document Frequency)
5. Calculate cosine similarity
6. Analyze the limitations of simple text representations


In [4]:
from pathlib import Path
file_path = Path("../data/sample_texts.txt")
with open(file_path, "r") as file:
    documents = file.readlines()

documents = [doc.strip() for doc in documents if doc.strip()]

for i, document in enumerate(documents):
    print(f"Document {i + 1}: {document}")

Document 1: The cat sat on the mat.
Document 2: The dog sat beside the cat.
Document 3: The bank approved the loan.
Document 4: She deposited money in the bank.
Document 5: They sat on the bank of the river.
Document 6: The river flowed beside the bank.
Document 7: Machine learning models learn patterns from data.
Document 8: Deep learning is a branch of machine learning.


In [5]:
import re

def tokenize(text):
    text = text.lower()
    tokens = re.findall(r"\b\w+\b", text)
    return tokens

sentence = documents[0]

tokens = tokenize(sentence)

print("Original sentence:")
print(sentence)
print("\nTokens:")
print(tokens)

Original sentence:
The cat sat on the mat.

Tokens:
['the', 'cat', 'sat', 'on', 'the', 'mat']


In [6]:
#Building a vocabulary: collection of unique words in our corpus.
vocabulary = set()
for document in documents:
    tokens = tokenize(document)
    vocabulary.update(tokens)

vocabulary = sorted(vocabulary)

print(vocabulary)
print(f"\nVocabulary size: {len(vocabulary)}")

['a', 'approved', 'bank', 'beside', 'branch', 'cat', 'data', 'deep', 'deposited', 'dog', 'flowed', 'from', 'in', 'is', 'learn', 'learning', 'loan', 'machine', 'mat', 'models', 'money', 'of', 'on', 'patterns', 'river', 'sat', 'she', 'the', 'they']

Vocabulary size: 29


In [7]:
#creating an index of the words
word_to_index = {
    word: index for index, word in enumerate(vocabulary)
    }
print(word_to_index)

{'a': 0, 'approved': 1, 'bank': 2, 'beside': 3, 'branch': 4, 'cat': 5, 'data': 6, 'deep': 7, 'deposited': 8, 'dog': 9, 'flowed': 10, 'from': 11, 'in': 12, 'is': 13, 'learn': 14, 'learning': 15, 'loan': 16, 'machine': 17, 'mat': 18, 'models': 19, 'money': 20, 'of': 21, 'on': 22, 'patterns': 23, 'river': 24, 'sat': 25, 'she': 26, 'the': 27, 'they': 28}


In [11]:
#implementing a Bag-of-Words (BoW) representation
import numpy as np

def bag_of_words(text, word_to_index):
    tokens = tokenize(text)

    vector = np.zeros(len(word_to_index), dtype=int)

    for token in tokens:
        if token in word_to_index:
            index = word_to_index[token]
            vector[index] += 1

    return vector

sentence = "The dog sat"

vector = bag_of_words(sentence, word_to_index)

print("Vocabulary:")
print(vocabulary)

print("\nSentence:")
print(sentence)

print("\nBag-of-Words Vector:")
print(vector)

Vocabulary:
['a', 'approved', 'bank', 'beside', 'branch', 'cat', 'data', 'deep', 'deposited', 'dog', 'flowed', 'from', 'in', 'is', 'learn', 'learning', 'loan', 'machine', 'mat', 'models', 'money', 'of', 'on', 'patterns', 'river', 'sat', 'she', 'the', 'they']

Sentence:
The dog sat

Bag-of-Words Vector:
[0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0]


In [13]:
#matrix to represent all documents in the corpus
bow_matrix = []

for document in documents:
    vector = bag_of_words(document, word_to_index)
    bow_matrix.append(vector)

bow_matrix = np.array(bow_matrix)

print("Shape:", bow_matrix.shape)
print(bow_matrix)

Shape: (8, 29)
[[0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 0 1 0 2 0]
 [0 0 0 1 0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 2 0]
 [0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 2 0]
 [0 0 1 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0 1 1 0]
 [0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 1 1 0 2 1]
 [0 0 1 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 2 0]
 [0 0 0 0 0 0 1 0 0 0 0 1 0 0 1 1 0 1 0 1 0 0 0 1 0 0 0 0 0]
 [1 0 0 0 1 0 0 1 0 0 0 0 0 1 0 2 0 1 0 0 0 1 0 0 0 0 0 0 0]]


## TF-IDF
Bag of Words treats all words as equally important.
TF-IDF improves this by assigning higher importance to words that occur frequently in a document but rarely across the entire corpus.

In [14]:
import math

def calculate_idf(documents, vocabulary):
    total_documents = len(documents)
    idf = {}

    for word in vocabulary:
        document_frequency = 0

        for document in documents:
            tokens = tokenize(document)
            if word in tokens:
                document_frequency += 1

        idf[word] = math.log(total_documents / (document_frequency + 1))

    return idf

idf_scores = calculate_idf(documents, vocabulary)
for word, score in idf_scores.items():
    print(f"Word: {word}, IDF: {score:.3f}")

Word: a, IDF: 1.386
Word: approved, IDF: 1.386
Word: bank, IDF: 0.470
Word: beside, IDF: 0.981
Word: branch, IDF: 1.386
Word: cat, IDF: 0.981
Word: data, IDF: 1.386
Word: deep, IDF: 1.386
Word: deposited, IDF: 1.386
Word: dog, IDF: 1.386
Word: flowed, IDF: 1.386
Word: from, IDF: 1.386
Word: in, IDF: 1.386
Word: is, IDF: 1.386
Word: learn, IDF: 1.386
Word: learning, IDF: 0.981
Word: loan, IDF: 1.386
Word: machine, IDF: 0.981
Word: mat, IDF: 1.386
Word: models, IDF: 1.386
Word: money, IDF: 1.386
Word: of, IDF: 0.981
Word: on, IDF: 0.981
Word: patterns, IDF: 1.386
Word: river, IDF: 0.981
Word: sat, IDF: 0.693
Word: she, IDF: 1.386
Word: the, IDF: 0.134
Word: they, IDF: 1.386


In [ ]:
def calculate_tf(text):
    tokens = tokenize(text)
    total_words = len(tokens)

    tf = {}
    for word in tokens:
        tf[word] = tf.get(word, 0) + 1

    # Normalize TF values
    for word in tf:
        tf[word] /= total_words

    return tf

sentence = "The cat sat on the mat"

tf_scores = calculate_tf(sentence)

print(scores)

{'the': 0.3333333333333333, 'cat': 0.16666666666666666, 'sat': 0.16666666666666666, 'on': 0.16666666666666666, 'mat': 0.16666666666666666}
